# Calendar Dimension

Builds the standard calendar table `adb_rapollocare.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31. Fiscal columns assume an April–March fiscal year (FY named by the year in which it ends).

In [0]:
CREATE OR REPLACE TABLE adb_rapollocare.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31. Fiscal year runs April-March and is named by its ending year.'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                AS date_key,
  d                                                      AS date,

  -- Day
  day(d)                                                 AS day_of_month,
  extract(DAYOFWEEK_ISO FROM d)                          AS day_of_week,        -- 1 = Monday ... 7 = Sunday
  date_format(d, 'EEEE')                                 AS day_name,
  date_format(d, 'EEE')                                  AS day_name_short,
  dayofyear(d)                                           AS day_of_year,
  CASE WHEN extract(DAYOFWEEK_ISO FROM d) IN (6, 7) THEN true ELSE false END AS is_weekend,
  CASE WHEN extract(DAYOFWEEK_ISO FROM d) IN (6, 7) THEN false ELSE true END AS is_weekday,

  -- Week (ISO 8601, weeks start Monday)
  weekofyear(d)                                          AS week_of_year,
  extract(YEAROFWEEK FROM d)                             AS iso_year,
  CAST(date_trunc('WEEK', d) AS DATE)                    AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)       AS week_end_date,
  CAST(ceil(day(d) / 7.0) AS INT)                        AS week_of_month,

  -- Month
  month(d)                                               AS month,
  date_format(d, 'MMMM')                                 AS month_name,
  date_format(d, 'MMM')                                  AS month_name_short,
  date_format(d, 'yyyy-MM')                              AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                  AS year_month_key,
  CAST(date_trunc('MONTH', d) AS DATE)                   AS month_start_date,
  last_day(d)                                            AS month_end_date,
  day(last_day(d))                                       AS days_in_month,
  d = CAST(date_trunc('MONTH', d) AS DATE)               AS is_month_start,
  d = last_day(d)                                        AS is_month_end,

  -- Quarter
  quarter(d)                                             AS quarter,
  concat('Q', quarter(d))                                AS quarter_name,
  concat(year(d), '-Q', quarter(d))                      AS year_quarter,
  CAST(date_trunc('QUARTER', d) AS DATE)                 AS quarter_start_date,
  date_add(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), -1) AS quarter_end_date,
  datediff(d, CAST(date_trunc('QUARTER', d) AS DATE)) + 1 AS day_of_quarter,

  -- Half year
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END              AS half_year,
  concat('H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END) AS half_year_name,

  -- Year
  year(d)                                                AS year,
  CAST(date_trunc('YEAR', d) AS DATE)                    AS year_start_date,
  make_date(year(d), 12, 31)                             AS year_end_date,
  (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 AS is_leap_year,
  d = CAST(date_trunc('YEAR', d) AS DATE)                AS is_year_start,
  d = make_date(year(d), 12, 31)                         AS is_year_end,

  -- Fiscal (April - March, FY named by ending year)
  year(add_months(d, 9))                                 AS fiscal_year,
  concat('FY', year(add_months(d, -3)), '-', right(CAST(year(add_months(d, 9)) AS STRING), 2)) AS fiscal_year_name,
  quarter(add_months(d, -3))                             AS fiscal_quarter,
  month(add_months(d, -3))                               AS fiscal_month
FROM dates;

In [0]:
SELECT
  count(*)                      AS row_count,
  count(DISTINCT date_key)      AS distinct_keys,
  min(date)                     AS min_date,
  max(date)                     AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_rows,
  sum(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS feb29_count
FROM adb_rapollocare.silver.calendar;

In [0]:
SELECT *
FROM adb_rapollocare.silver.calendar
WHERE date IN (DATE'1940-01-01', DATE'2024-02-29', DATE'2024-03-31', DATE'2024-04-01', DATE'2026-11-10', DATE'2030-12-31')
ORDER BY date;